In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
from colorama import Fore, Style

In [ ]:
# Data

data = ""
with open('data/shakespeare.txt', 'r') as sh:
    for line in sh.readlines():
        data += line
def preprocess(text):
    text = text.strip().lower()
    text = text.replace("\n\n", "\n")
    for (redund, symb) in {
        "1": "0",
        "2": "0",
        "3": "0",
        "4": "0",
        "5": "0",
        "6": "0",
        "7": "0",
        "8": "0",
        "9": "0",
    }.items(): text = text.replace(redund, symb)
    return text
data = replace_nums(text)

chars = np.unique([d for d in data])
decoder = dict(enumerate(chars))
tokenizer = {decoder[i]: i for i in decoder.keys()}

sequences = []
targets = []
seq_length = 40
for x in range(len(data) - seq_length):
    sequences += [[d for d in data[x:x+seq_length]]]
    targets += [data[x+40]]
if len(sequences[-1]) != seq_length:
    sequences = sequences[:-1]

print(len(sequences))

In [ ]:
# Tokenization

X = torch.tensor([[tokenizer[d] for d in seq] for seq in sequences], dtype=int)
y = torch.tensor([tokenizer[d] for d in targets], dtype=int)
print(X.shape, y.shape)

In [ ]:
# Architecture

class William(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim, temperature=1):
        super(William, self).__init__()

        self.D = 1
        self.H_c = hidden_dim
        self.H_i = input_dim 
        self.H_o = output_dim 
        self.N_LAYERS = layer_dim
        self.T = temperature

        self.embedding = nn.Embedding(self.H_o, self.H_i)

        self.lstm = nn.LSTM(
            input_size=self.H_i,
            hidden_size=self.H_c,
            num_layers=self.N_LAYERS,
            batch_first=True,
            proj_size=0
        )
        self.full = nn.Linear(self.H_c, self.H_o)

    def forward(self, x, h0=None, c0=None):
        h0 = torch.zeros(self.D * self.N_LAYERS, x.shape[0], self.H_c)
        c0 = torch.zeros(self.D * self.N_LAYERS, x.shape[0], self.H_c)

        x = self.embedding(x)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.full(out[:, -1, :])/self.T  # Take last time step
        return out, hn, cn # logits, hidden, cell

In [ ]:
# Training

batch_size = 1000
seq_length = 40
input_dim = 64
hidden_dim = 150  # Use for cell and hidden state if proj_size = 0
layer_dim = 1     # Use for hidden_size if proj_size > 0
output_dim = len(tokenizer)

bill = William(
    input_dim,
    hidden_dim,
    layer_dim,
    output_dim,
)

criterion = nn.CrossEntropyLoss()
learning_rate = 0.003
optimizer = optim.Adam(bill.parameters(), lr=learning_rate)

n_epochs = 35
h0, c0 = None, None
increment = X.shape[0] // batch_size
for epoch in range(n_epochs):
    epoch_loss = 0
    for i in range(increment):
        idx = i*batch_size
    
        bill.train()
        optimizer.zero_grad()

        outputs, hn, cn = bill(X[idx:idx+batch_size], (h0, c0))

        loss = criterion(outputs, y[idx:idx+batch_size])
        loss.backward()
        optimizer.step()

        # h0, c0 = hn.detach(), cn.detach()
        epoch_loss += loss.item()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}, Loss: {epoch_loss/increment:.4f}')

# epochs, datapoints, embedding dimension, hidden dimension
MODEL_PATH = f"bill{n_epochs}-d{X.shape[0]}-e{input_dim}-h{hidden_dim}.pt"
torch.save(bill, MODEL_PATH)

In [ ]:
# Generation

i = 80
seed_text = preprocess("shall i compare thee to a summer's day?\n") # preprocess(data[i:i+40])
generate_length = 120
temperatures = list(range(0.5, 2, 0.5))

# model = torch.load("bill50-d97970-e64-h150.pt", weights_only=False)
# model.eval()

current_seq = [tokenizer[c] for c in seed_text]

print(Style.BRIGHT + f"\033[4mSeed Text: Green; Generated Text: Red\033[0m")
for temperature in temperatures:
    seed_text = "shall i compare thee to a summer's day?\n"
    generated_text = ""
    with torch.no_grad():
        for _ in range(generate_length):
            x_input = torch.tensor([current_seq])
            
            logits = bill(x_input)[0]
            probs = torch.softmax(logits/temperature, dim=1).squeeze()
            
            next_char_idx = torch.multinomial(probs, 1).item()      
            generated_text += decoder[next_char_idx]
            
            # Slide the window (remove first char, append predicted char)
            current_seq = current_seq[1:] + [next_char_idx]

    print(f"Temperature {temperature}:")
    print(Style.RESET_ALL + Fore.GREEN + seed_text + Fore.RED + generated_text + Style.RESET_ALL + "\n")